In [2]:
import os
import json
import fitz  # PyMuPDF
import pdfplumber
import pandas as pd
from PIL import Image
from pathlib import Path
from langchain_core.documents import Document

In [ ]:
"""
                                       File Paths
"""

PDF_PATH = r"D:\RAG\DataParsing\doc\complex_rag_parsing_sample_with_sunny_image.pdf"

OUTPUT_DIR = Path("D:\RAG\DataParsing\doc\output")

IMAGE_DIR = OUTPUT_DIR / "extracted_images"
PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("PDF exists:", os.path.exists(PDF_PATH))

In [5]:
# ============================================================
#  Below is code to convert images to text using python module pytessercat
# ============================================================

def run_ocr_on_image(image_path):
    """
    Runs OCR on image using pytesseract.
    If tesseract is not installed in system, it will return empty text.
    """
    try:
        import pytesseract
        pytesseract.pytesseract.tesseract_cmd = (r"C:\Program Files\Tesseract-OCR\tesseract.exe")
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return f"[OCR_SKIPPED_OR_FAILED: {str(e)}]"

In [6]:

#  Below is the python code to extract text, images from pdf (pdf can have multiple pages)

def extract_text_images_from_pdf(pdf_path):

        doc = fitz.open(PDF_PATH)  # fitz will open the pdf, type of doc is pymupdf.Document but not Langchain document.
        page_records = []
        image_records = []

        for page_index in range(len(doc)):

        #                         Exctracting text from each pdf page.                                           #
                page = doc[page_index]
                page_number  = page_index + 1
                text = page.get_text("text")    # This extracts only text that is already stored in the PDF but not images, charts....

                page_info = {

                                "page_number": page_number,
                                "text": text.strip(),
                                "image_count": len(page.get_images(full=True)),
                                "page_width": page.rect.width,
                                "page_height": page.rect.height
                }

                page_records.append(page_info)


                                                                                                                
                #                Covert pdf page into image and saved to local path.                                           
                

                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                page_image_path = PAGE_IMAGE_DIR / f"page_{page_number:03d}.png"
                pix.save(str(page_image_path))


                
                #                Now we need to run OCR for the images saved to extract the text.

                
                ocr_text = run_ocr_on_image(page_image_path)
                page_info["ocr_text"] = ocr_text
                page_info["page_image_path"] = str(page_image_path)


                
                #       Below is the python code to Extract embedded iamges from pdf page


                images = page.get_images(full=True) 

                for image_index, img in enumerate(images): # here img_index = 0, img = (12, 0, 500, 300, ...)

                        xref = img[0]     # Every image embedded in a PDF has a unique cross-reference number (xref).
                                        # It is like image's unique ID inside the PDF.

                        base_image = doc.extract_image(xref)    # Exctract actual image from xref
                                                                # Now, base image is a dictinary which contain 
                                                                # information of images like
                                                                # {
                                                                # "image": b"...binary bytes...",
                                                                #        "ext": "png",
                                                                #        "width": 500,
                                                                #        "height": 300,
                                                                #        "colorspace": 3
                                                                #        }

                        image_bytes = base_image["image"]       # This is image content in raw binary data
                        image_ext = base_image["ext"]           # Image extention

                        image_path = IMAGE_DIR / f"page_{page_number:03d}_image_{image_index + 1}.{image_ext}"
                        with open(image_path, "wb") as f:
                                f.write(image_bytes)
                        image_ocr_text = run_ocr_on_image(image_path)  # To extract text from images using OCR

                        image_records.append({
                                "page_number": page_number,
                                "image_index": image_index + 1,
                                "image_path": str(image_path),
                                "image_ext": image_ext,
                                "image_ocr_text": image_ocr_text
                        })


        doc.close()

        return page_records, image_records



page_records, image_records = extract_text_images_from_pdf(PDF_PATH)
print("Total pages parsed:", len(page_records))
print("Total images extracted:", len(image_records))


Total pages parsed: 24
Total images extracted: 13


In [ ]:

#  Below is the python code to extract tables from pdf (pdf can have multiple pages)

def extracted_tables_from_pdf(PDF_PATH):
    table_records=[]

    with pdfplumber.open(PDF_PATH) as pdf:

        for page_index, page in enumerate(pdf.pages):

            page_number = page_index + 1

            try:
                tables = page.extract_tables()
            except Exception as e:
                tables = []
                print(f"Table extraction Failed on page {page_number}: {e}")

            for table_index, table in enumerate(tables):
                if not table:
                    continue

                cleaned_table = []
                for row in table:
                    cleaned_row = [
                            cell.strip() if isinstance(cell, str) else cell
                            for cell in row
                        ]
                    cleaned_table.append(cleaned_row)


                # convert each table to pandas Dataframe
                try:
                    df = pd.DataFrame(cleaned_table[1:], columns=cleaned_table[0])
                except:
                    df = pd.DataFrame(cleaned_table)

                table_records.append({

                    "page_number": page_number,
                    "table_index": table_index + 1,
                    "raw_table": cleaned_table,
                    "markdown": df.to_markdown(index=False),
                    "csv":df.to_csv(index=False)
                })

    return table_records            


table_records = extracted_tables_from_pdf(PDF_PATH)
print("Total tables extracted:", len(table_records))

for table in table_records[:3]:
    print("\nPage:", table["page_number"], "Table:", table["table_index"])
    print(table["markdown"][:1000])





In [55]:
# Converting the text_records, image_records and table_records to a Langchain Documents


langchain_docs = []


for page in page_records:
    
    # Coverting pdf text to langchain document
    
    page_number = page["page_number"]

    combined_text = f""" PAGE {page_number}

                         SELCTABLE TEXT {page["text"]}

                         OCR_TEXT {page["ocr_text"]}
                    
    """.strip()

    doc = Document(

                    page_content=combined_text,

                    metadata= {
                                "source": PDF_PATH,
                                "page_number": page_number,
                                "content_type": "page_text_plus_ocr",
                                "image_count": page["image_count"],
                                "page_image_path": page["page_image_path"],
                    }
    )

    langchain_docs.append(doc)


# Converting table in PDF to Langchain Document


for table in table_records:


    page_number = table["page_number"]

    table_text = f""" TABLE FOUND ON PAGE {table["page_number"]}

                      TABLE INDEX {table["table_index"]}

                      TABLE MARKDOWN {table["markdown"]} """.strip()
    

    doc = Document(

           page_content= table_text,

           metadata = {
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "table",
            "table_index": table["table_index"]
           }
    )

    langchain_docs.append(doc)



# Converting images in PDF to Langchain Document

for image in image_records:

    page_number = image["page_number"]

    image_text = f"""
                        IMAGE FOUND ON PAGE {page_number}

                        IMAGE INDEX: {image["image_index"]}
                        
                        IMAGE PATH: {image["image_path"]}
                        
                        IMAGE OCR TEXT: {image["image_ocr_text"]}""".strip()
    
    doc = Document(

        page_content= image_text,

        metadata = {
            "source": PDF_PATH,
            "page_number": page_number,
            "content_type": "image",
            "image_index": image["image_index"],
            "image_path": image["image_path"],
            "image_ext": image["image_ext"],
        }

    )

    langchain_docs.append(doc)

print("Total LangChain Documents created:", len(langchain_docs))


Total LangChain Documents created: 53


In [57]:
# Saving page_records, image_records, table_records into JSON


# Save page records
with open(OUTPUT_DIR / "page_records.json", "w", encoding="utf-8") as f:
    json.dump(page_records, f, indent=2, ensure_ascii=False)

# Save image records
with open(OUTPUT_DIR / "image_records.json", "w", encoding="utf-8") as f:
    json.dump(image_records, f, indent=2, ensure_ascii=False)

# Save table records
with open(OUTPUT_DIR / "table_records.json", "w", encoding="utf-8") as f:
    json.dump(table_records, f, indent=2, ensure_ascii=False)

# Save all LangChain document content as markdown
with open(OUTPUT_DIR / "rag_ready_documents.md", "w", encoding="utf-8") as f:
    for i, doc in enumerate(langchain_docs):
        f.write(f"\n\n# Document {i + 1}\n")
        f.write(f"\nMetadata:\n```json\n{json.dumps(doc.metadata, indent=2)}\n```\n")
        f.write("\nContent:\n")
        f.write(doc.page_content)
        f.write("\n\n---\n")

# Save table markdown separately
with open(OUTPUT_DIR / "extracted_tables.md", "w", encoding="utf-8") as f:
    for table in table_records:
        f.write(f"\n\n## Page {table['page_number']} - Table {table['table_index']}\n\n")
        f.write(table["markdown"])
        f.write("\n\n---\n")

print("\nSaved outputs in:", OUTPUT_DIR)


Saved outputs in: D:\RAG\Data
